Aggregate per-year storm exceedance metrics (from `storm_percentile_metrics.py`) over the full 35-year period for each simulation. Storms spanning two calendar years are summed.

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [2]:
simulations = ['UBD', 'UBE', 'UBF', 'UBG', 'UBH', 'UBI']
future_hist_sim = {'UBG': 'UBD', 'UBH': 'UBE', 'UBI': 'UBF'}
quantiles = [99.0, 99.9]
wetdays = False
future = False
mask = 'urban'

In [3]:
base = '/home/vdemeyer/projects/rrg-gachon/vdemeyer'

quantile_tag = {99.0: '99', 99.9: '999'}
mtag = 'urbanonly' if mask == 'urban' else 'land_only'

for sim in simulations:

    # Determine period
    if sim in future_hist_sim:
        years = range(2063, 2098)
    else:
        years = range(1980, 2015)

    # Build suffix for filenames
    add_file = ''
    if wetdays:
        add_file += '_wetdays'
    if future and sim in future_hist_sim:
        add_file += '_future_percentile'

    

    per_quantile_dfs = []

    for q in quantiles:
        tag = quantile_tag[q]

        # Load all per-year pickles for this quantile
        dfs = []
        for year in tqdm(years, desc=f'{sim} q={q}'):
            pkl_file = f'{base}/{sim}/STORM_RELATED/STORM_METRICS/storm_exceed_{tag}_metrics_{mtag}_{sim}_{year}{add_file}.pkl'
            if not os.path.exists(pkl_file):
                print(f"Warning: missing {pkl_file}")
                continue
            dfs.append(pd.read_pickle(pkl_file))

        if not dfs:
            print(f"No pickle files found for {sim} q={q}, skipping")
            continue

        # Concatenate and sum cum/count for storms appearing in multiple years
        df_all = pd.concat(dfs, ignore_index=True)
        df_agg = df_all.groupby('storm_id', as_index=False).agg({
            'cum_excess_pr': 'sum',
            'count_exceed_pr': 'sum',
            'cum_excess_wind': 'sum',
            'count_exceed_wind': 'sum',
        })

        # Compute intensity = cum / count
        df_agg['mean_excess_pr'] = np.where(
            df_agg['count_exceed_pr'] > 0,
            df_agg['cum_excess_pr'] / df_agg['count_exceed_pr'],
            np.nan
        )
        df_agg['mean_excess_wind'] = np.where(
            df_agg['count_exceed_wind'] > 0,
            df_agg['cum_excess_wind'] / df_agg['count_exceed_wind'],
            np.nan
        )

        df_agg['quantile'] = q
        per_quantile_dfs.append(df_agg)

    if not per_quantile_dfs:
        continue

    df_agg = pd.concat(per_quantile_dfs, ignore_index=True)

    # Load tracking data and extract initiation date per storm
    storm_data_file = f'{base}/{sim}/STORM_RELATED/TRACK/{sim}_psl_smooth_400km_24h_1000hPa.txt'
    df_track = pd.read_csv(
        storm_data_file,
        sep=r' ', header=0, engine='python',
        names=['storm', 'point', 'i', 'j', 'date', 'lat', 'lon', 'pressure']
    )
    df_track['date'] = pd.to_datetime(df_track['date'])
    initiation = df_track.groupby('storm')['date'].min().reset_index()
    initiation.columns = ['storm_id', 'initiation_date']

    # Merge initiation date
    df_agg = df_agg.merge(initiation, on='storm_id', how='left')

    # Filter: keep only storms whose initiation falls within the period
    start_year, end_year = min(years), max(years)
    df_agg = df_agg[
        (df_agg['initiation_date'].dt.year >= start_year) &
        (df_agg['initiation_date'].dt.year <= end_year)
    ]

    # Reorder columns
    df_agg = df_agg[['storm_id', 'quantile', 'initiation_date',
                      'cum_excess_pr', 'count_exceed_pr', 'mean_excess_pr',
                      'cum_excess_wind', 'count_exceed_wind', 'mean_excess_wind']]
    df_agg = df_agg.sort_values(['quantile', 'storm_id']).reset_index(drop=True)

    # Save
    output_dir = f'{base}/{sim}/STORM_RELATED/STORM_METRICS'
    os.makedirs(output_dir, exist_ok=True)
    output_file = f'{output_dir}/storm_exceed_metrics_{mtag}_{sim}_{start_year}-{end_year}{add_file}.pkl'
    df_agg.to_pickle(output_file)
    print(f"\n{sim}: saved {len(df_agg)} rows to {output_file}")
    for q in quantiles:
        sub = df_agg[df_agg['quantile'] == q]
        print(f"  q={q}: {len(sub)} storms | pr exceedance: {(sub['count_exceed_pr'] > 0).sum()} | wind exceedance: {(sub['count_exceed_wind'] > 0).sum()}")

UBD q=99.9: 100%|██████████| 35/35 [00:01<00:00, 32.79it/s]



UBD: saved 18840 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBD/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_urbanonly_UBD_1980-2014.pkl
  q=99.0: 9420 storms | pr exceedance: 7506 | wind exceedance: 6671
  q=99.9: 9420 storms | pr exceedance: 5034 | wind exceedance: 3974


UBE q=99.9: 100%|██████████| 35/35 [00:01<00:00, 29.39it/s]



UBE: saved 18200 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBE/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_urbanonly_UBE_1980-2014.pkl
  q=99.0: 9100 storms | pr exceedance: 7341 | wind exceedance: 6579
  q=99.9: 9100 storms | pr exceedance: 4941 | wind exceedance: 4024


UBF q=99.9: 100%|██████████| 35/35 [00:01<00:00, 25.23it/s]


...


UBG: saved 17132 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBG/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_urbanonly_UBG_2063-2097.pkl
  q=99.0: 8566 storms | pr exceedance: 6966 | wind exceedance: 5885
  q=99.9: 8566 storms | pr exceedance: 4966 | wind exceedance: 3563


UBH q=99.9: 100%|██████████| 35/35 [00:01<00:00, 23.98it/s]



UBH: saved 17882 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBH/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_urbanonly_UBH_2063-2097.pkl
  q=99.0: 8941 storms | pr exceedance: 7144 | wind exceedance: 6235
  q=99.9: 8941 storms | pr exceedance: 5011 | wind exceedance: 3784


UBI q=99.9: 100%|██████████| 35/35 [00:00<00:00, 67.43it/s]



UBI: saved 19472 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBI/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_urbanonly_UBI_2063-2097.pkl
  q=99.0: 9736 storms | pr exceedance: 7809 | wind exceedance: 6831
  q=99.9: 9736 storms | pr exceedance: 5627 | wind exceedance: 4016
